# 14 - Unsupervised models: validation threshold and new data

1. Pick the PCA / Isolation Forest threshold on a validation part instead of the test set.
2. Find out why Isolation Forest fails on HDFS.
3. Run Isolation Forest on the unlabelled OpenSSH and Apache windows.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.evaluation.metrics import evaluate
from src.models.unsupervised import (IForestDetector, PCADetector, TfidfZeroMean, fit_and_score, score_unlabelled,
                                     threshold_sweep, time_split, top_flagged, validated_results)

pd.set_option("display.max_colwidth", 200)
DOCS, PROCESSED = ROOT / "docs", ROOT / "data" / "processed"
PERCENTILES = [90, 95, 97.5, 99, 99.5]

hdfs = pd.read_parquet(PROCESSED / "hdfs_counts.parquet")
bgl = pd.read_parquet(PROCESSED / "bgl_counts.parquet")

## Isolation Forest on HDFS: why it fails

Same setup as notebook 06: normal blocks of the first 70% for training, the last 30% for testing.

In [2]:
event_cols = list(hdfs.filter(regex=r"^E\d+$").columns)
train, test = time_split(hdfs, train_frac=0.7)
X_train = train.loc[train["Label"] == 0, event_cols].to_numpy()
X_test, y_test = test[event_cols].to_numpy(), test["Label"].to_numpy()

detector = IForestDetector().fit(X_train)
scores = detector.score(X_test)
print(f"Training labels: {np.unique(train.loc[train['Label'] == 0, 'Label'])}   rows per tree (max_samples): {detector.model.max_samples_}")
print(f"Mean score: normal {scores[y_test == 0].mean():.3f}, anomalous {scores[y_test == 1].mean():.3f}   ROC-AUC {roc_auc_score(y_test, scores):.3f}")

Training labels: [0]   rows per tree (max_samples): 256
Mean score: normal 0.432, anomalous 0.479   ROC-AUC 0.685


Score direction and training data are fine: anomalies score higher on average, and only normal blocks are used. The count vectors explain the problem:

In [3]:
vectors, copies = np.unique(X_train, axis=0, return_counts=True)
seen = {v.tobytes() for v in vectors}
unseen = X_train.sum(axis=0) == 0
has_unseen = X_test[:, unseen].sum(axis=1) > 0

print(f"Distinct count vectors in {len(X_train):,} normal training blocks: {len(vectors)}")
print(f"Share of the most common vector: {copies.max() / len(X_train):.1%}")
print(f"Anomalous test blocks identical to a normal training block: {np.mean([v.tobytes() in seen for v in X_test[y_test == 1]]):.1%}")
print(f"Events always 0 in normal training: {unseen.sum()} of {len(event_cols)}")
print(f"Test blocks with such an event: anomalous {has_unseen[y_test == 1].mean():.1%}, normal {has_unseen[y_test == 0].mean():.1%}")

Distinct count vectors in 389,427 normal training blocks: 209
Share of the most common vector: 51.7%
Anomalous test blocks identical to a normal training block: 0.0%
Events always 0 in normal training: 26 of 45
Test blocks with such an event: anomalous 27.7%, normal 0.0%


- With the default 256 rows per tree, about half of each tree's sample is copies of one vector, and the rare normal shapes are missing. Rare normal blocks then look as isolated as anomalies.
- Trees can't split on an event that is always 0 in training, so the 28% of anomalies with such an event are invisible to the model.

Variants (best F1 over the percentiles, test set, for diagnosis only):

In [4]:
tfidf = TfidfZeroMean().fit(X_train)
variants = {
    "raw counts, max_samples=256 (old)": (IForestDetector(), X_train, X_test),
    "TF-IDF features": (IForestDetector(), tfidf.transform(X_train), tfidf.transform(X_test)),
    "raw counts, max_samples=1024": (IForestDetector(max_samples=1024), X_train, X_test),
    "raw counts, max_samples=all": (IForestDetector(max_samples=1.0), X_train, X_test),
    "raw counts, max_samples=all + unseen events": (IForestDetector(max_samples=1.0, flag_unseen=True), X_train, X_test),
}
rows = []
for name, (model, A, B) in variants.items():
    train_scores, test_scores, fit_s, _ = fit_and_score(model, A, B)
    sweep = pd.DataFrame(threshold_sweep(name, train_scores, test_scores, y_test, PERCENTILES))
    rows.append({"variant": name, "roc_auc": roc_auc_score(y_test, test_scores), "best f1": sweep["f1"].max(), "fit_seconds": fit_s})
pd.DataFrame(rows).style.format({"roc_auc": "{:.3f}", "best f1": "{:.3f}", "fit_seconds": "{:.1f}"}).hide(axis="index")

variant,roc_auc,best f1,fit_seconds
"raw counts, max_samples=256 (old)",0.685,0.051,0.3
TF-IDF features,0.680,0.037,0.3
"raw counts, max_samples=1024",0.741,0.256,0.5
"raw counts, max_samples=all",0.888,0.484,3.1
"raw counts, max_samples=all + unseen events",0.957,0.654,3.3


Fix: `max_samples=1.0` (each tree sees all training rows) plus `flag_unseen=True` (blocks with an event never seen in training score above every normal block). TF-IDF does not help.

## Validation split

First 60% for fitting (normal rows only), next 10% for validation, last 30% for testing (the same test set as before). The threshold percentile is the one with the best F1 on the validation part.

In [5]:
def make_detectors():
    return {"PCA": PCADetector(variance=0.95),
            "Isolation Forest (old)": IForestDetector(),
            "Isolation Forest (fixed)": IForestDetector(max_samples=1.0, flag_unseen=True)}


val_tables, test_tables = {}, {}
for name, counts, order_col in [("HDFS", hdfs, "order"), ("BGL", bgl, "window")]:
    val_tables[name], test_tables[name] = validated_results(counts, make_detectors(), PERCENTILES, order_col)

chosen = pd.concat(test_tables, names=["dataset"]).reset_index(level=0)
chosen.round(4).to_csv(DOCS / "unsupervised_validated_results.csv", index=False)
chosen[["dataset", "model", "percentile", "val_f1", "precision", "recall", "f1", "false_alarm_rate"]].style.format(
    {c: "{:.3f}" for c in ["precision", "recall", "f1", "false_alarm_rate"]} | {"val_f1": "{:.3f}", "percentile": "{:g}"}).hide(axis="index")

dataset,model,percentile,val_f1,precision,recall,f1,false_alarm_rate
HDFS,PCA,95,0.854,0.860,0.817,0.838,0.003
HDFS,Isolation Forest (old),90,0.053,0.043,0.032,0.037,0.016
HDFS,Isolation Forest (fixed),99,0.719,0.787,0.511,0.620,0.003
BGL,PCA,99,0.258,0.095,0.980,0.173,0.755
BGL,Isolation Forest (old),90,0.033,0.042,0.046,0.044,0.085
BGL,Isolation Forest (fixed),99.5,0.276,0.096,0.991,0.175,0.752


Validation F1 for every percentile:

In [6]:
pd.concat(val_tables, names=["dataset"]).reset_index(level=0).pivot_table(
    index=["dataset", "model"], columns="percentile", values="f1", sort=False).round(3)

percentile                         90.0   95.0   97.5   99.0   99.5
dataset model                                                      
HDFS    PCA                       0.689  0.854  0.854  0.854  0.854
        Isolation Forest (old)    0.053  0.053  0.007  0.004  0.003
        Isolation Forest (fixed)  0.245  0.689  0.689  0.719  0.719
BGL     PCA                       0.217  0.242  0.254  0.258  0.258
        Isolation Forest (old)    0.033  0.029  0.026  0.022  0.021
        Isolation Forest (fixed)  0.240  0.258  0.274  0.276  0.276

Old tables (notebooks 06 and 10: fitted on the full 70%, every percentile evaluated on the test set), for comparison:

In [7]:
old = pd.concat({name: pd.read_csv(DOCS / f"{name.lower()}_unsupervised_results.csv") for name in ["HDFS", "BGL"]},
                names=["dataset"]).reset_index(level=0)
old.pivot_table(index=["dataset", "model"], columns="percentile", values="f1", sort=False).round(3)

percentile                 90.0   95.0   97.5   99.0   99.5
dataset model                                              
HDFS    PCA               0.644  0.644  0.836  0.838  0.854
        Isolation Forest  0.051  0.051  0.007  0.002  0.002
BGL     PCA               0.240  0.290  0.307  0.311  0.312
        Isolation Forest  0.038  0.037  0.021  0.004  0.002

## Isolation Forest on OpenSSH and Apache

No labels. The first 70% of windows (in time order) are assumed normal for training; the last 30% are scored. Same fixed Isolation Forest as above. Thresholds are the 95th and 99th percentiles of the training scores.

In [8]:
flag_rates, flagged = [], {}
for name in ["openssh", "apache"]:
    counts = pd.read_parquet(PROCESSED / f"{name}_counts.parquet")
    sequences = pd.read_parquet(PROCESSED / f"{name}_sequences.parquet").set_index("window")
    templates = pd.read_csv(PROCESSED / f"{name}_templates.csv").set_index("EventId")["template"]

    detector = IForestDetector(max_samples=1.0, flag_unseen=True)
    test, test_scores, train_scores = score_unlabelled(counts, detector)
    flag_rates.append({"dataset": name, "test windows": len(test),
                       "with unseen event": (test_scores > 1).mean(),
                       **{f"flagged at p{p}": (test_scores > np.percentile(train_scores, p)).mean() for p in [95, 99]}})
    windows = sequences.loc[test["window"]].reset_index()
    event_cols = counts.filter(regex=r"^E\d+$").columns
    train_events = set(event_cols[counts.iloc[:len(counts) - len(test)][event_cols].sum() > 0])
    flagged[name] = top_flagged(windows, test_scores, templates, n=5, n_events=3, known_events=train_events)

pd.DataFrame(flag_rates).style.format({c: "{:.1%}" for c in ["with unseen event", "flagged at p95", "flagged at p99"]}).hide(axis="index")

dataset,test windows,with unseen event,flagged at p95,flagged at p99
openssh,4046,1.0%,14.6%,3.0%
apache,848,0.1%,0.6%,0.5%


Top 5 flagged OpenSSH windows (1 minute each) and their 3 most frequent events. A score above 1 means the window has an event never seen in training (last column).

In [9]:
flagged["openssh"].style.format({"score": "{:.3f}"}, escape="html").hide(axis="index")

window,score,lines,top events,unseen events
12125,1.426,80,E2 x9: Invalid user <*> from <IP>; E3 x9: input_userauth_request: invalid user <*> [preauth]; E4 x9: pam_unix(sshd:auth): check pass; user unknown,E49: Received disconnect from <IP>: <NUM>: Disconnect requested by Windows SSH Client.
10709,1.361,66,E11 x11: message repeated <NUM> times: [ Failed password for <USER> from <IP> port <PORT> ssh2]; E12 x11: Disconnecting: Too many authentication failures for <*> <*>; E13 x11: PAM <NUM> more authentication <*> logname= uid=<NUM> euid=<NUM> tty=ssh ruser= <*> user=<USER>,E49: Received disconnect from <IP>: <NUM>: Disconnect requested by Windows SSH Client.
12809,1.331,56,"E6 x10: Failed <*> for invalid user <*> from <IP> port <PORT> ssh2; E50 x10: Received disconnect from <IP>: <NUM>: Normal Shutdown, Thank you for <USER> [preauth]; E2 x9: Invalid user <*> from <IP>","E50: Received disconnect from <IP>: <NUM>: Normal Shutdown, Thank you for <USER> [preauth]"
12820,1.320,13,E2 x2: Invalid user <*> from <IP>; E3 x2: input_userauth_request: invalid user <*> [preauth]; E4 x2: pam_unix(sshd:auth): check pass; user unknown,"E50: Received disconnect from <IP>: <NUM>: Normal Shutdown, Thank you for <USER> [preauth]"
12830,1.320,13,E2 x2: Invalid user <*> from <IP>; E3 x2: input_userauth_request: invalid user <*> [preauth]; E4 x2: pam_unix(sshd:auth): check pass; user unknown,"E50: Received disconnect from <IP>: <NUM>: Normal Shutdown, Thank you for <USER> [preauth]"


Top 5 flagged Apache windows (20 lines each):

In [10]:
flagged["apache"].style.format({"score": "{:.3f}"}, escape="html").hide(axis="index")

window,score,lines,top events,unseen events
2198,1.326,20,E30 x8: [client <IP>] client sent HTTP/<NUM>.<NUM> request without hostname (see RFC2616 section <NUM>.<NUM>): /; E14 x6: [client <IP>] Directory index forbidden by rule: /var/www/html/; E15 x4: [client <IP>] File does not exist: <*>,E30: [client <IP>] client sent HTTP/<NUM>.<NUM> request without hostname (see RFC2616 section <NUM>.<NUM>): /
2182,0.666,20,"E27 x14: child process <NUM> still did not exit, sending a SIGTERM; E1 x2: LDAP: Built with OpenLDAP LDAP SDK; E2 x1: LDAP: SSL support unavailable",
2183,0.624,20,E11 x8: jk2_init() Found child <NUM> in scoreboard slot <NUM>; E6 x4: env.createBean2(): Factory error creating <*> ( <*> <*>; E7 x4: config.update(): Can't create <*>,
2027,0.605,20,E6 x4: env.createBean2(): Factory error creating <*> ( <*> <*>; E7 x4: config.update(): Can't create <*>; E16 x3: jk2_init() Can't find child <NUM> in scoreboard,
2026,0.442,20,E19 x19: mod_jk2 Shutting down; E4 x1: Digest: generating secret for digest authentication ...,
